# Preparing - Load library `aleph`

## Experiment configuration

In [ ]:
SEEDS = [11111, 22222, 33333, 44444, 55555]
DATASETS = [
    "QX_Lung",
    "QX_Marrow",
    "QS_Heart",
    "QS_Lung",
    "QS_Diaphragm",
    "QS_Limb_Muscle",
]

from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "topology_signature.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from inside the TopoCLR repository.")

BASE_DIR = PROJECT_ROOT / "data"  # Folder containing the local .h5 files.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TOPO_WEIGHT = 10

DISTANCE_TYPE = "euclidean"
loss_type = "MSE"

class Config:
    cuda = True
    seed = SEEDS[0]
    epoch = 2
    select_gene = 2000
    batch_size = 64
    dropout = 0.9
    lr = 0.01
    m = 0.5
    noise = 0.1
    temperature = 0.07
    enc_1 = 200
    enc_2 = 40
    enc_3 = 60
    mlp_dim = 40
    cluster_methods = ["KMeans"]

args = Config()

RUN_ROBUSTNESS = False
ROBUSTNESS_DATASET = "QS_Diaphragm"  # Used only when RUN_ROBUSTNESS is True.

## Load library

In [ ]:
import os
import sys
import random
import time
import math
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy as sp
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import pandas as pd

from IPython.display import display

def load_expression(path):
    with h5py.File(path, "r") as data_file:
        counts = np.asarray(data_file["X"])
        labels = np.asarray(data_file["Y"]).reshape(-1)
    if counts.ndim != 2 or counts.shape[0] != labels.size or not labels.size:
        raise ValueError("Expected a count matrix and one label per cell.")
    return counts, labels


device = torch.device('cuda:0' if torch.cuda.is_available() and args.cuda else 'cpu')

In [ ]:
from utils.preprocessing import normalize

# --- Utilities ---
def get_device(use_cpu):
    if use_cpu is None:
        if torch.cuda.is_available():
            device = torch.device('cuda:0')
        else:
            device = torch.device('cpu')
    else:
        device = torch.device('cpu')
    return device
    
def preprocess(gene_exp, select_genes, *, labels=None):
    X = np.ceil(gene_exp).astype(np.float32) 
    
    print(f"Original shape: {X.shape}, keeping {select_genes} genes")
    adata = sc.AnnData(X)
    if labels is not None:
        adata.obs["reference_label"] = labels

    adata = normalize(adata, copy=True, highly_genes=select_genes,
                             size_factors=True, normalize_input=True, logtrans_input=True)
    
    X = adata.X.astype(np.float32)
    if labels is not None:
        return X, adata.obs["reference_label"].to_numpy()
    return X

def adjust_learning_rate(optimizer, epoch, lr):
    eta_min = lr * (0.1 ** 3)
    new_lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / 500)) / 2
    for param_group in optimizer.param_groups:
        param_group['lr'] = new_lr
    return lr

def save_model(name, model, optimizer, current_epoch, pre_epoch):
    save_dir = os.path.join(os.getcwd(), "save", name)
    os.makedirs(save_dir, exist_ok=True)
    if pre_epoch != -1:
        pre_path = os.path.join(save_dir, "checkpoint_{}.tar".format(pre_epoch))
        if os.path.exists(pre_path):
            os.remove(pre_path)
            
    cur_path = os.path.join(save_dir, "checkpoint_{}.tar".format(current_epoch))
    state = {'net': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': current_epoch}
    torch.save(state, cur_path)

def cluster_embedding(embedding, cluster_number, real_label, save_pred=False, cluster_methods=None):
    if cluster_methods is None: cluster_methods = ["KMeans"]
    result = {"t_clust": time.time()}
    if "KMeans" in cluster_methods:
        kmeans = KMeans(n_clusters=cluster_number, init="k-means++", random_state=0)
        pred = kmeans.fit_predict(embedding)
        if real_label is not None:
            result["ari"] = round(adjusted_rand_score(real_label, pred), 4)
            result["nmi"] = round(normalized_mutual_info_score(real_label, pred), 4)
        result["t_k"] = time.time()
        if save_pred: result["pred"] = pred
    return result

# --- Losses ---
class InstanceLoss(nn.Module):
    def __init__(self, temperature):
        super(InstanceLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

    def mask_correlated_samples(self, batch_size):
        N = 2 * batch_size
        mask = torch.ones((N, N)).fill_diagonal_(0)
        for i in range(batch_size):
            mask[i, batch_size + i] = 0
            mask[batch_size + i, i] = 0
        return mask.bool()

    def forward(self, features):
        self.batch_size = features.shape[0]
        self.mask = self.mask_correlated_samples(self.batch_size).to(device)
        N = 2 * self.batch_size
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)

        sim = torch.matmul(contrast_feature, contrast_feature.T) / self.temperature
        sim_i_j = torch.diag(sim, self.batch_size)
        sim_j_i = torch.diag(sim, -self.batch_size)
        
        positive_samples = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(N, 1)
        negative_samples = sim[self.mask].reshape(N, -1)

        labels = torch.zeros(N).to(positive_samples.device).long()
        logits = torch.cat((positive_samples, negative_samples), dim=1)

        loss = self.criterion(logits, labels) / N
        return loss

class ClusterLoss(nn.Module):
    def __init__(self, class_num, temperature):
        super(ClusterLoss, self).__init__()
        self.class_num = class_num
        self.temperature = temperature
        self.mask = self.mask_correlated_clusters(class_num)
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

    def mask_correlated_clusters(self, class_num):
        N = 2 * class_num
        mask = torch.ones((N, N)).fill_diagonal_(0)
        for i in range(class_num):
            mask[i, class_num + i] = 0
            mask[class_num + i, i] = 0
        return mask.bool()

    def forward(self, features_cluster):
        c_i, c_j = torch.unbind(features_cluster, dim=1)
        p_i = c_i.sum(0).view(-1)
        p_i /= p_i.sum()
        ne_i = math.log(p_i.size(0)) + (p_i * torch.log(p_i + 1e-8)).sum()
        
        p_j = c_j.sum(0).view(-1)
        p_j /= p_j.sum()
        ne_j = math.log(p_j.size(0)) + (p_j * torch.log(p_j + 1e-8)).sum()
        ne_loss = ne_i + ne_j

        N = 2 * self.class_num
        cluster_feature = torch.cat(torch.unbind(features_cluster, dim=1), dim=0)
        cluster_feature = torch.nn.functional.normalize(cluster_feature, p=2, dim=1)
        sim = torch.matmul(cluster_feature, cluster_feature.T) / self.temperature
        sim_i_j = torch.diag(sim, self.class_num)
        sim_j_i = torch.diag(sim, -self.class_num)

        positive_clusters = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(N, 1)
        negative_clusters = sim[self.mask].reshape(N, -1)

        labels = torch.zeros(N).to(positive_clusters.device).long()
        logits = torch.cat((positive_clusters, negative_clusters), dim=1)
        loss = self.criterion(logits, labels) / N

        return loss + ne_loss

In [ ]:
class DataAug(nn.Module):
    def __init__(self, dropout=0.9):
        super(DataAug, self).__init__()
        self.drop = nn.Dropout(p=dropout)

    def forward(self, x):
        return self.drop(x)

class BaseEncoder(nn.Module):
    def __init__(self, dims):
        super(BaseEncoder, self).__init__()
        self.dims = dims
        enc = [nn.Linear(self.dims[0], self.dims[1]), nn.BatchNorm1d(self.dims[1]), nn.ReLU(),
               nn.Linear(self.dims[1], self.dims[2]), nn.BatchNorm1d(self.dims[2]), nn.ReLU(),
               nn.Linear(self.dims[2], self.dims[3]), nn.BatchNorm1d(self.dims[3]), nn.ReLU()]
        self.encoder = nn.Sequential(*enc)
        self._reset_prams()

    def _reset_prams(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        latent_out = self.encoder(x)
        return F.normalize(latent_out, dim=1)

class MLP(nn.Module):
    def __init__(self, contrastive_dim, mlp_dim, dim):
        super(MLP, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(contrastive_dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(self, x):
        return self.encoder(x)

class ScCCL(nn.Module):
    def __init__(self, encoder_q, encoder_k, instance_projector, cluster_projector, class_num, m=0.2):
        super(ScCCL, self).__init__()
        self.m = m
        self.encoder_q = encoder_q
        self.encoder_k = encoder_k
        
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data.copy_(param_q.data)
            param_k.requires_grad = False

        self.instance_projector = instance_projector
        self.cluster_projector = nn.Sequential(cluster_projector, nn.Softmax(dim=1))

    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

    def forward(self, cell_q, cell_k):
        q = self.encoder_q(cell_q)
        q_instance = F.normalize(self.instance_projector(q), dim=1)
        q_cluster = self.cluster_projector(q)

        if cell_k is None:
            return q_instance, q_cluster, None, None

        with torch.no_grad():
            self._momentum_update_key_encoder()
            k = self.encoder_k(cell_k)
            k_instance = F.normalize(self.instance_projector(k), dim=1)
            k_cluster = self.cluster_projector(k)

        return q_instance, q_cluster, k_instance, k_cluster

In [ ]:
from sklearn.decomposition import PCA
from topology_signature import TopologySignatureLoss

# --- Main ScCCL Training Logic ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# EPOCH_NOTIFY_EVERY = 10
EPOCH_NOTIFY_EVERY = 50

def run(gene_exp, cluster_number, dataset, real_label, epochs, lr, temperature, dropout, layers, batch_size, m,
        save_pred=True, noise=None, use_cpu=None, cluster_methods=None, topo_weight=10):
    if cluster_methods is None:
        cluster_methods = []
    results = {}

    start = time.time()
    embedding, best_model, model = train_model(gene_exp=gene_exp, cluster_number=cluster_number, real_label=real_label,
                                        epochs=epochs, lr=lr, temperature=temperature,
                                        dropout=dropout, layers=layers, batch_size=batch_size,
                                        m=m, save_pred=save_pred, noise=noise, use_cpu=use_cpu, topo_weight=topo_weight)

    if save_pred:
        results[f"features"] = embedding
        results[f"max_epoch"] = best_model
    elapsed = time.time() - start
    res_eval = cluster_embedding(embedding, cluster_number, real_label, save_pred=save_pred,
                                 cluster_methods=cluster_methods)
    results = {**results, **res_eval, "dataset": dataset, "time": elapsed}

    return results


def train_model(gene_exp, cluster_number, real_label, epochs, lr,
                temperature, dropout, layers, batch_size, m,
                save_pred=False, noise=None, use_cpu=None, evaluate_training=True, topo_weight=10, *, seed=None):
    
    set_seed(args.seed if seed is None else seed)
    device = get_device(use_cpu)

    dims = np.concatenate([[gene_exp.shape[1]], layers])
    data_aug_model = DataAug(dropout=dropout)
    encoder_q = BaseEncoder(dims)
    encoder_k = BaseEncoder(dims)
    instance_projector = MLP(layers[2], layers[2] + layers[3], layers[2] + layers[3])
    cluster_projector = MLP(layers[2], layers[3], cluster_number)
    model = ScCCL(encoder_q, encoder_k, instance_projector, cluster_projector, cluster_number, m=m)
    data_aug_model.to(device)
    model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    criterion_instance = InstanceLoss(temperature=temperature)
    criterion_cluster = ClusterLoss(cluster_number, temperature=temperature)



    topo_criterion = TopologySignatureLoss(max_dimension=1, loss_type=loss_type, distance_type=DISTANCE_TYPE).to(device)

    for epoch in range(epochs):
        model.train()
        adjust_learning_rate(optimizer, epoch, lr)
    
        apply_topo = epoch >= (epochs // 2) and topo_weight > 0

        loss_instance_ = 0
        loss_cluster_ = 0
        loss_topo_ = 0

        # Random permutation of the data for batching
        idx = np.random.permutation(len(gene_exp))

        # Moving into main training loop with batch processing
        for pre_index in range(len(gene_exp) // batch_size + 1):
            c_idx = np.arange(pre_index * batch_size,
                              min(len(gene_exp), (pre_index + 1) * batch_size))
            if len(c_idx) < 2:
                continue
            
            c_idx = idx[c_idx]
            c_inp = gene_exp[c_idx]

            
            input1 = data_aug_model(torch.FloatTensor(c_inp))
            input2 = data_aug_model(torch.FloatTensor(c_inp))

            if noise is None or noise == 0:
                input1 = torch.FloatTensor(input1).to(device)
                input2 = torch.FloatTensor(input2).to(device)
            else:
                noise_vec = torch.FloatTensor(np.random.normal(loc=0, scale=noise, size=input1.shape))
                input1 = torch.FloatTensor(input1 + noise_vec).to(device)
                noise_vec = torch.FloatTensor(np.random.normal(loc=0, scale=noise, size=input2.shape))
                input2 = torch.FloatTensor(input2 + noise_vec).to(device)
            
            q_instance, q_cluster, k_instance, k_cluster = model(input1, input2)

            # Contrastive Instance and Cluster Losses
            features_instance = torch.cat([q_instance.unsqueeze(1), k_instance.unsqueeze(1)], dim=1)
            features_cluster = torch.cat([q_cluster.t().unsqueeze(1), k_cluster.t().unsqueeze(1)], dim=1)
            
            loss_instance = criterion_instance(features_instance)
            loss_cluster = criterion_cluster(features_cluster)

            loss = loss_instance + loss_cluster
            
            if apply_topo:
                t_loss_inst = topo_criterion.forward_by_batch(q_instance, k_instance)
                t_loss_clust = topo_criterion.forward_by_batch(q_cluster, k_cluster)

                batch_topo_loss = topo_weight * (t_loss_inst + t_loss_clust)
                
                loss += batch_topo_loss
                loss_topo_ += batch_topo_loss.item()
            
            loss_instance_ += loss_instance.item()
            loss_cluster_ += loss_cluster.item()


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Log only the loss progress during training
        if epoch == 0 or (epoch + 1) % EPOCH_NOTIFY_EVERY == 0:
            total_loss = loss_instance_ + loss_cluster_ + (loss_topo_ if apply_topo else 0)
            topo_str = f"{loss_topo_:.4f}" if apply_topo else "0.000"
            print(f"Epoch {epoch + 1}/{epochs}: Loss: {total_loss:.4f}, Topo: {topo_str}, Instance: {loss_instance_:.4f}, Cluster: {loss_cluster_:.4f}")

    # --- TRAINING COMPLETE: Evaluate only on the final epoch ---
    model.eval()
    with torch.no_grad():
        q_instance, _, _, _ = model(torch.FloatTensor(gene_exp).to(device), None)
        features = q_instance.detach().cpu().numpy()
        
    if evaluate_training and real_label is not None:
        res = cluster_embedding(features, cluster_number, real_label, save_pred=save_pred)
        print(f"\n--- Final Epoch {epochs} Results ---")
        print(f"ARI: {res['ari']}, NMI: {res['nmi']}")

    # Save the final model state
    best_model = epochs - 1

    return features, best_model, model

In [ ]:
def prepare_dataset(dataset):
    counts, labels = load_expression(BASE_DIR / f"{dataset}.h5")
    features, labels = preprocess(counts, args.select_gene, labels=labels)
    if len(labels) < 2:
        raise ValueError("At least two cells are required after preprocessing.")
    num_clusters = np.unique(labels).size
    return features, labels, num_clusters


def train_and_evaluate(prepared, seed, weight, keep_model):
    features, labels, num_clusters = prepared
    embedding, _, model = train_model(
        gene_exp=features, cluster_number=num_clusters, real_label=labels,
        epochs=args.epoch, lr=args.lr, temperature=args.temperature,
        dropout=args.dropout, layers=[args.enc_1, args.enc_2, args.enc_3, args.mlp_dim],
        batch_size=args.batch_size, m=args.m, noise=args.noise,
        use_cpu=None if args.cuda else True, evaluate_training=False, topo_weight=weight, seed=seed,
    )
    predictions = KMeans(
        n_clusters=num_clusters, init="k-means++", random_state=seed
    ).fit_predict(embedding)
    return (
        normalized_mutual_info_score(labels, predictions),
        adjusted_rand_score(labels, predictions),
        model if keep_model else None,
    )

In [ ]:
TRAINING_MODES = {"Without topology": 0, "With topology": TOPO_WEIGHT}

if RUN_ROBUSTNESS and ROBUSTNESS_DATASET not in DATASETS:
    raise ValueError("ROBUSTNESS_DATASET must be included in DATASETS.")
if args.epoch < 1 or TOPO_WEIGHT <= 0:
    raise ValueError("Use a positive topology loss weight and at least one epoch.")

def run_experiments():
    scores = []
    selected_pair = None
    for dataset in DATASETS:
        print(f"\nDataset: {dataset}", flush=True)
        try:
            prepared = prepare_dataset(dataset)
        except Exception as error:
            print(f"{dataset}: preparation failed ({type(error).__name__}: {error})", flush=True)
            continue

        for seed in SEEDS:
            keep_models = RUN_ROBUSTNESS and dataset == ROBUSTNESS_DATASET and selected_pair is None
            models = []
            for mode, weight in TRAINING_MODES.items():
                run_label = f"{dataset} | seed={seed} | {mode}"
                print(run_label, flush=True)
                try:
                    random.seed(seed)
                    np.random.seed(seed)
                    torch.manual_seed(seed)
                    nmi, ari, model = train_and_evaluate(prepared, seed, weight, keep_models)
                    if not np.isfinite([nmi, ari]).all():
                        raise FloatingPointError("Non-finite clustering score.")
                except Exception as error:
                    print(f"{run_label}: failed ({type(error).__name__}: {error})", flush=True)
                    continue

                scores.append((dataset, seed, mode, nmi, ari))
                if keep_models:
                    models.append(model)

            if keep_models and len(models) == len(TRAINING_MODES):
                selected_pair = (prepared, models)
                print(f"Using {dataset} | seed={seed} for robustness evaluation.", flush=True)


    results = pd.DataFrame(
        scores, columns=["Dataset", "Seed", "Mode", "NMI", "ARI"]
    ).astype({"NMI": "float64", "ARI": "float64"})
    return results, selected_pair


run_results, robustness_pair = run_experiments()

## NMI and ARI by dataset

Mean and sample standard deviation (SD, ddof=1) are shown when every configured seed succeeds for that dataset and mode.

In [ ]:
summary = run_results.groupby(["Dataset", "Mode"]).agg(
    completed_seeds=("Seed", "count"),
    NMI_mean=("NMI", "mean"),
    NMI_SD=("NMI", "std"),
    ARI_mean=("ARI", "mean"),
    ARI_SD=("ARI", "std"),
)
expected_rows = pd.MultiIndex.from_product(
    [DATASETS, TRAINING_MODES], names=["Dataset", "Mode"]
)
summary = summary.reindex(expected_rows).rename(columns={
    "completed_seeds": "Completed seeds",
    "NMI_mean": "NMI mean",
    "NMI_SD": "NMI SD",
    "ARI_mean": "ARI mean",
    "ARI_SD": "ARI SD",
})
summary["Completed seeds"] = summary["Completed seeds"].fillna(0).astype(int)

incomplete = summary["Completed seeds"] != len(SEEDS)
metric_columns = ["NMI mean", "NMI SD", "ARI mean", "ARI SD"]
summary.loc[incomplete, metric_columns] = np.nan
display(summary.reset_index().round(4))

## Optional robustness evaluation

Set RUN_ROBUSTNESS to True and choose a ROBUSTNESS_DATASET from DATASETS.
The robustness evaluation uses the first seed that completes both training modes for that dataset.

In [ ]:
if RUN_ROBUSTNESS:
    from utils.robustness_topology import (
        DEFAULT_LEVELS, DEFAULT_SEEDS, make_default_appliers, run_robustness_comparison,
    )
    from utils.robustness_clustering import run_clustering_robustness_comparison

    ROBUSTNESS_LEVELS = DEFAULT_LEVELS
    ROBUSTNESS_SEEDS = DEFAULT_SEEDS
    # For a quick smoke run: ROBUSTNESS_LEVELS = (0.0, 0.1); ROBUSTNESS_SEEDS = (33333,)
    ROBUSTNESS_OUTPUT_DIR = PROJECT_ROOT / "results" / "robustness"

    def _torch_encoder(model, select_instance=False):
        def encode(x, aux):
            model.eval()
            model_device = next(model.parameters()).device
            x = torch.as_tensor(x, dtype=torch.float32, device=model_device)
            with torch.no_grad():
                z = model(x, None)[0] if select_instance else model(x)
            return z.detach().cpu()
        return encode


    if robustness_pair is None:
        raise RuntimeError(f"No seed completed both modes for {ROBUSTNESS_DATASET}.")
    prepared, fitted_models = robustness_pair

    gene_exp, real_label, _ = prepared
    model_1, model_2 = fitted_models

    pert_list = ["feature_dropout", "gaussian_noise"]
    ROBUSTNESS_APPLIERS = make_default_appliers(pert_list)
    models_for_robustness = {
        label: dict(get_representation_fn=_torch_encoder(model, select_instance=True),
                    X_clean=torch.as_tensor(gene_exp, dtype=torch.float32), aux_clean=None,
                    perturbation_appliers=ROBUSTNESS_APPLIERS)
        for label, model in [("Without Topology", model_1), ("With Topology", model_2)]
    }
    robustness_labels = np.asarray(real_label).reshape(-1)

    robustness_num_clusters = len(np.unique(robustness_labels))
    robustness_dataset = ROBUSTNESS_DATASET
    robustness_file_prefix = "scccl" + "_" + Path(robustness_dataset).name

In [ ]:
if RUN_ROBUSTNESS:
    topology_results = run_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_topology.csv"),
    )
    display(topology_results.head())

In [ ]:
if RUN_ROBUSTNESS:
    clustering_results = run_clustering_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        Y_true=robustness_labels, num_clusters=robustness_num_clusters,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_clustering.csv"),
    )
    display(clustering_results.head())